# 🎨 Playground 2: Pix2Pix – Bút Vẽ Phù Thủy (Sketch-to-Art)
### Biến các nét vẽ phác thảo nguệch ngoạc thành tác phẩm nghệ thuật đầy màu sắc và đổ bóng

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDanh1510/GAN-playground/blob/main/notebooks/02_pix2pix_sketch2art_playground.ipynb)

---

## 🎯 Mục Tiêu Bài Học:
1. Hiểu kiến trúc **U-Net Generator**: Tại sao các đường nối tắt (*Skip Connections*) lại cực kỳ quan trọng để giữ nét viền phác thảo.
2. Hiểu **PatchGAN Discriminator**: Cách mạng nơ-ron chia nhỏ ảnh thành các ô vuông nhỏ $N \times N$ để chấm điểm từng chi tiết.
3. Kết hợp 2 hàm mất mát: **Adversarial Loss ($L_{GAN}$)** để ảnh sắc nét + **L1 Loss ($L_{L1}$)** để màu sắc trung thực.

### 1. Cài đặt môi trường & Kiểm tra GPU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Thiết bị: {device}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")

### 2. Xây dựng U-Net Generator (Encoder - Decoder với Skip Connections)

```
Input (Sketch) --> [Encoder e1] ---------------------> [Decoder d3] --> Output (Color Image)
                      \                                   / (Skip Connection)
                    [Encoder e2] ------------> [Decoder d2]
                        \                         /
                       [Encoder e3] --> [Decoder d1]
```

In [ ]:
class MiniUNetGenerator(nn.Module):
    def __init__(self, in_c=1, out_c=3):
        super().__init__()
        # Encoder (Trích xuất đặc trưng nét vẽ)
        self.e1 = nn.Conv2d(in_c, 32, 4, 2, 1) # 64 -> 32
        self.e2 = nn.Sequential(nn.Conv2d(32, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.e3 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        
        # Decoder (Tô màu chi tiết)
        self.d1 = nn.Sequential(nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU())
        self.d2 = nn.Sequential(nn.ConvTranspose2d(64 + 64, 32, 4, 2, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU())
        self.d3 = nn.Sequential(nn.ConvTranspose2d(32 + 32, out_c, 4, 2, 1), nn.Tanh())
        
    def forward(self, x):
        e1 = nn.LeakyReLU(0.2)(self.e1(x))
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        
        d1 = self.d1(e3)
        d2 = self.d2(torch.cat([d1, e2], dim=1)) # Ghép nối Skip Connection
        out = self.d3(torch.cat([d2, e1], dim=1)) # Ghép nối Skip Connection
        return out

# 3. PatchGAN Discriminator
class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=4): # 1 kênh Sketch + 3 kênh Color
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 1, 4, 1, 1),
            nn.Sigmoid()
        )
    def forward(self, sketch, color):
        x = torch.cat([sketch, color], dim=1)
        return self.net(x)

print("✓ Khởi tạo MiniUNetGenerator và PatchGANDiscriminator thành công!")

### 3. Tạo tập dữ liệu phác thảo hình học nhân tạo (Geometric Art Paired Dataset)

In [ ]:
def create_paired_batch(batch_size=16, img_size=64):
    sketches = torch.zeros(batch_size, 1, img_size, img_size)
    colors = torch.zeros(batch_size, 3, img_size, img_size)
    for i in range(batch_size):
        cx, cy = np.random.randint(20, 44, 2)
        r = np.random.randint(10, 18)
        color = np.random.rand(3) * 2 - 1 # Chuẩn hóa [-1, 1]
        
        y, x = np.ogrid[:img_size, :img_size]
        dist = np.sqrt((x - cx)**2 + (y - cy)**2)
        # Nét phác thảo viền
        sketches[i, 0, (dist >= r - 2) & (dist <= r + 2)] = 1.0
        # Tranh tô màu hoàn chỉnh
        for c in range(3):
            colors[i, c, dist <= r] = color[c]
    return sketches, colors

# Hiển thị mẫu dữ liệu huấn luyện
demo_s, demo_c = create_paired_batch(4)
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    axes[0, i].imshow(demo_s[i, 0], cmap='gray'); axes[0, i].set_title(f"Nét vẽ {i+1}"); axes[0, i].axis('off')
    axes[1, i].imshow(((demo_c[i].permute(1, 2, 0) + 1) / 2).clip(0, 1)); axes[1, i].set_title(f"Ảnh màu {i+1}"); axes[1, i].axis('off')
plt.suptitle("Cặp Dữ Liệu Huấn Luyện Pix2Pix (Sketches ↔ Colors)")
plt.show()

### 4. Huấn luyện Pix2Pix (U-Net + PatchGAN với Loss $L_{cGAN} + 100 \times L_{L1}$)

In [ ]:
epochs = 60
batch_size = 16
lambda_l1 = 100.0

G_pix = MiniUNetGenerator().to(device)
D_pix = PatchGANDiscriminator().to(device)

bce_loss = nn.BCELoss()
l1_loss = nn.L1Loss()

opt_G = optim.Adam(G_pix.parameters(), lr=0.002, betas=(0.5, 0.999))
opt_D = optim.Adam(D_pix.parameters(), lr=0.002, betas=(0.5, 0.999))

print("⚡ Đang bắt đầu huấn luyện Pix2Pix...")
for epoch in range(1, epochs + 1):
    sketches, real_colors = create_paired_batch(batch_size)
    sketches = sketches.to(device)
    real_colors = real_colors.to(device)
    
    # --- Train Cảnh Sát D ---
    fake_colors = G_pix(sketches)
    pred_real = D_pix(sketches, real_colors)
    pred_fake = D_pix(sketches, fake_colors.detach())
    
    loss_D = (bce_loss(pred_real, torch.ones_like(pred_real) * 0.9) +
              bce_loss(pred_fake, torch.zeros_like(pred_fake))) / 2
    opt_D.zero_grad(); loss_D.backward(); opt_D.step()
    
    # --- Train Máy Tạo G ---
    pred_fake_g = D_pix(sketches, fake_colors)
    loss_gan = bce_loss(pred_fake_g, torch.ones_like(pred_fake_g))
    loss_l1 = l1_loss(fake_colors, real_colors) * lambda_l1
    loss_G = loss_gan + loss_l1
    
    opt_G.zero_grad(); loss_G.backward(); opt_G.step()
    
    if epoch % 15 == 0 or epoch == epochs:
        print(f"Epoch [{epoch:02d}/{epochs}] | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f} (L1: {loss_l1.item():.2f})")

print("✓ Huấn luyện Pix2Pix hoàn tất!")

### 5. Thử tài Bút Vẽ Phù Thủy: Kiểm tra ảnh do AI tô màu!

In [ ]:
test_sketches, test_reals = create_paired_batch(4)
with torch.no_grad():
    test_fakes = G_pix(test_sketches.to(device)).cpu()

fig, axes = plt.subplots(3, 4, figsize=(11, 8))
for i in range(4):
    axes[0, i].imshow(test_sketches[i, 0], cmap='gray'); axes[0, i].set_title(f"Nét vẽ đầu vào {i+1}"); axes[0, i].axis('off')
    axes[1, i].imshow(((test_fakes[i].permute(1, 2, 0) + 1) / 2).clamp(0, 1)); axes[1, i].set_title(f"✨ AI Tô Màu (U-Net)"); axes[1, i].axis('off')
    axes[2, i].imshow(((test_reals[i].permute(1, 2, 0) + 1) / 2).clamp(0, 1)); axes[2, i].set_title(f"Ảnh Mẫu Thực Tế"); axes[2, i].axis('off')

plt.suptitle("Kết Quả Thử Nghiệm Pix2Pix Sketch-to-Art", fontsize=14, fontweight='bold')
plt.show()